In [2]:
from datasets import load_dataset

flirt = load_dataset("shirshatzman/flirtflip-dataset")

c:\Users\LENOVO\miniconda3\envs\gpt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(flirt)
print(flirt["train"][0])

DatasetDict({
    train: Dataset({
        features: ['original', 'gentle', 'playful', 'bold', 'id', 'scenario'],
        num_rows: 1071
    })
})
{'original': 'Great choice!', 'gentle': "Great choice! You clearly have excellent taste... in everything, I'm guessing", 'playful': 'Great choice! Though between you and me, the second best choice today was me coming over to talk to you', 'bold': 'Great choice! Now make another one and give me your number', 'id': 'flirt_0001', 'scenario': 'shared activity'}


In [4]:
for i in range(5):
    print(flirt["train"][i])

{'original': 'Great choice!', 'gentle': "Great choice! You clearly have excellent taste... in everything, I'm guessing", 'playful': 'Great choice! Though between you and me, the second best choice today was me coming over to talk to you', 'bold': 'Great choice! Now make another one and give me your number', 'id': 'flirt_0001', 'scenario': 'shared activity'}
{'original': 'Is this seat taken?', 'gentle': "Is this seat taken? I promise I'm good company", 'playful': 'Is this seat taken, or were you saving it for someone devastatingly charming? Because here I am!', 'bold': "This seat's about to be taken... by me. Hope you don't mind the best view in the place", 'id': 'flirt_0002', 'scenario': 'day activity'}
{'original': 'Nice weather today', 'gentle': "Beautiful day, isn't it? Though I have to say, you're the brightest thing I've seen all morning", 'playful': 'Nice weather, right? Or did you just bring the sunshine with you?', 'bold': "Forget the weather - you're the only forecast I'm inte

In [5]:
conversations = []

for example in flirt["train"]:
    user_msg = example['original']

    for style in ["gentle", "playful", "bold"]:
        assistant_msg = example[style]
        conversations.append(
            f"<|user|>\n{user_msg}\n"
            f"<|assistant|>\n{assistant_msg}\n"
        )

print("Number of conversations:", len(conversations))
print()
print(conversations[:5])

Number of conversations: 3213

["<|user|>\nGreat choice!\n<|assistant|>\nGreat choice! You clearly have excellent taste... in everything, I'm guessing\n", '<|user|>\nGreat choice!\n<|assistant|>\nGreat choice! Though between you and me, the second best choice today was me coming over to talk to you\n', '<|user|>\nGreat choice!\n<|assistant|>\nGreat choice! Now make another one and give me your number\n', "<|user|>\nIs this seat taken?\n<|assistant|>\nIs this seat taken? I promise I'm good company\n", '<|user|>\nIs this seat taken?\n<|assistant|>\nIs this seat taken, or were you saving it for someone devastatingly charming? Because here I am!\n']


In [6]:
import re

all_text = "\n".join(conversations)

print("Total characters:", len(all_text))
print("Total conversations:", len(conversations))
print("Average characters/conversation:", len(all_text) / len(conversations))

words = re.findall(r"\S+", all_text)

print("Approximate word count:", len(words))
print("Unique words:", len(set(words)))

Total characters: 327245
Total conversations: 3213
Average characters/conversation: 101.85029567382509
Approximate word count: 51460
Unique words: 1821


In [7]:
lengths = [len(x) for x in conversations]

print("Shortest:", min(lengths))
print("Longest:", max(lengths))
print("Average:", sum(lengths) / len(lengths))

Shortest: 51
Longest: 151
Average: 100.85060690943044


In [8]:
import random
random.seed(42)
random.shuffle(conversations)

split_idx = int(0.9 * len(conversations))

train_conversations = conversations[:split_idx]
val_conversations = conversations[split_idx:]

train_text = '\n'.join(train_conversations)
val_text = '\n'.join(val_conversations)

print("Train:", len(train_conversations), len(train_text), "chars")
print("Val:", len(val_conversations), len(val_text), "chars")


Train: 2891 294518 chars
Val: 322 32726 chars


In [9]:
chars = sorted(list(set(train_text + val_text)))

vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print("Vocabulary size:", vocab_size)
print(chars)

Vocabulary size: 67
['\n', ' ', '!', '%', "'", ',', '-', '.', '0', '1', '2', '3', '5', ':', '<', '>', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|']


In [10]:
batch_size = 32
block_size = 128

n_embd = 128
n_head = 4
n_layer = 6

dropout = 0.1

learning_rate = 3e-4
max_iters = 10000

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [13]:
# single head self attention
class Head(nn.Module): # Pytorch nn module

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size))) # Added causal mask

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # calculate scaled attention scores
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)

        # applied causal mask - prevent tokens from looking into the future
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # convert scores into attention weights
        wei = F.softmax(wei, dim=-1)

        # weighted aggregation of value vector
        out = wei @ v
        return out

In [14]:
# multi-head attention
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        self.proj = nn.Linear(n_embd, n_embd) # Linear projection to mix and transform all the o/p of each heads.
        dropout = 0.0
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [15]:
class FeedForward(nn.Module):
    """a simple linear layer followed by a non-linearity"""
    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout()
        )

    def forward(self, x):
        return self.net(x)

In [16]:
class Block(nn.Module):
    """Transformer block: communication followed by computation"""
    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # Layer Norm -> Multi Head Attention -> Residual Connection -> Layer Norm -> ffwd nn -> Residual Connection
        x = x + self.ffwd(self.ln2(x))
        return x

In [50]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head) for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        if targets == None:
            loss = None

        else:
            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=20):
        for _ in range(max_new_tokens):

            # Keep only the last block_size tokens as context
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # We only care about the prediction for the last token
            logits = logits[:, -1, :]

            # Temperature
            logits = logits / temperature

            # Top-k sampling
            if top_k is not None:
                values, indices = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < values[:, [-1]]] = float("-inf")

            # Convert logits into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append the new token to the sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

# Architecture
#-------------------------------
# Characters
#     ↓
# Token IDs
#     ↓
# Token Embeddings
#     +
# Position Embeddings
#     ↓
# 4 Transformer Blocks
#     │
#     ├── Multi-Head Self-Attention
#     └── Feed-Forward Network
#     ↓
# Final LayerNorm
#     ↓
# Linear LM Head
#     ↓
# 65 logits
#     ↓
# Cross-Entropy Loss


In [53]:
model = GPTLanguageModel().to(device)

print(
    f"Parameters: "
    f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

Parameters: 1.22M


In [56]:
checkpoint = torch.load("flirtgpt.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

print("Trained FlirtGPT restored! 🚀")

Trained FlirtGPT restored! 🚀


In [19]:
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)

def get_batch(split):
    data = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [20]:
for iter in range(max_iters):

    xb, yb = get_batch("train")

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 1000 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 4.3750
step 1000: loss 1.2756
step 2000: loss 0.8147
step 3000: loss 0.6781
step 4000: loss 0.4838
step 5000: loss 0.4197
step 6000: loss 0.3259
step 7000: loss 0.2580
step 8000: loss 0.3017
step 9000: loss 0.2434


In [57]:
def generate_response(prompt, max_new_tokens=200):
    context = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    generated = model.generate(
        context,
        max_new_tokens=max_new_tokens,
        temperature=0.8,
        top_k=20
    )

    text = decode(generated[0].tolist())

    response = text.rsplit("<|assistant|>\n", 1)[-1]
    response = response.split("<|user|>", 1)[0]

    return response.strip()


In [36]:
response = generate_response("<|user|>\nHey, how was your day?\n<|assistant|>\n")
print(response)

How was your day? Would've been better if you say we continue this privately?


In [58]:
conversation = ""
print("FlirtGPT\n")
print("type quit to exit!\n")

while True:
    user_msg = input("You: ")
    print("You: ", user_msg)
    if user_msg.lower() == "quit":
        print("bye")
        break
    conversation +=  f"<|user|>\n{user_msg}\n<|assistant|>\n"

    response = generate_response(conversation)
    print("FlirtGPT: ", response, "\n")

    conversation += response + "\n"


FlirtGPT

type quit to exit!

You:  hey there
FlirtGPT:  Dance with me. That wasn't a question very else 

You:  how are you
FlirtGPT:  You're worpose... think absolutely what when I met since and great 

You:  iam seriously bored right now
FlirtGPT:  Another dance?... I know what's my place are 

You:  you bitch
FlirtGPT:  Your eyes... want to be my be dreams 

You:  quit
bye


In [43]:
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab_size": vocab_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "block_size": block_size,
    "chars": chars,
}, "flirtgpt.pt")

print("Model saved successfully!")

Model saved successfully!
